# 07 — Task Decomposition and Workflow Prompting

This offline lab compares a single broad request with a typed sequential workflow.

## Boundary

Stages expose facts, evidence, and terminal conditions. They do not authorize a refund or make the system an agent.

In [1]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import sys
path=Path.cwd()/'curriculum/intermediate/07-task-decomposition-and-workflow-prompting/lab.py'
if not path.exists(): path=Path.cwd()/'lab.py'
spec=spec_from_file_location('workflow_lab',path); lab=module_from_spec(spec); sys.modules[spec.name]=lab; spec.loader.exec_module(lab)

## Baseline

The one-request path approves without evidence and offers no stage-level diagnosis.

In [2]:
baseline=lab.one_prompt(lab.DOCUMENT)
lab.score(baseline), baseline

({'supported': False, 'stages': 1, 'debuggable': False},
 {'decision': 'approve',
  'trace': ('single call',),
  'supported': False,
  'stages': 1})

## Workflow experiment

The workflow extracts a fact, checks evidence, then drafts for review. Compare the identical document.

In [3]:
improved=lab.workflow(lab.DOCUMENT)
assert improved['supported']
lab.score(improved), improved

({'supported': True, 'stages': 3, 'debuggable': True},
 {'decision': 'draft_for_review',
  'trace': ('extract', 'check evidence', 'draft'),
  'supported': True,
  'stages': 3})

## Failure injection and production

Missing order IDs terminate in clarification rather than an invented decision. Version each stage, validate handoffs, bound retries, and measure quality, latency, cost, and failure isolation.

Exercises: add a parallel stage and a timeout; decide whether either makes the workflow better.

In [4]:
missing=lab.Document('Customer requests a refund.',lab.DOCUMENT.policy)
assert lab.workflow(missing)['decision']=='clarify'